In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Resizing, RandomFlip, Lambda, Dense
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.datasets import cifar10

CLASS_IDS = [0, 1, 9]
CLASS_NAMES = ["airplane", "automobile", "truck"]
tf.keras.utils.set_random_seed(42)

2026-10-27 12:19:39.698116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train, y_test = y_train.ravel(), y_test.ravel()

train_keep = np.isin(y_train, CLASS_IDS)
test_keep = np.isin(y_test, CLASS_IDS)
x_train, y_train = x_train[train_keep], y_train[train_keep]
x_test, y_test = x_test[test_keep], y_test[test_keep]

label_map = {0: 0, 1: 1, 9: 2}
y_train = np.array([label_map[y] for y in y_train])
y_test = np.array([label_map[y] for y in y_test])
print("Training images:", len(x_train))
print("Test images:", len(x_test))

Training images: 15000
Test images: 3000


In [3]:
base_model = VGG16(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)
base_model.trainable = False

model = Sequential([
    Input(shape=(32, 32, 3)),
    Resizing(96, 96),
    RandomFlip("horizontal"),
    Lambda(preprocess_input),
    base_model,
    Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 512)            │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,716,227 (56.14 MB)

 Trainable params: 1,539 (6.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [4]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32
)

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy:.2%}")

Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 544s 1s/step - accuracy: 0.8097 - loss: 0.9352 - val_accuracy: 0.8967 - val_loss: 0.4080
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 460s 1s/step - accuracy: 0.9119 - loss: 0.3113 - val_accuracy: 0.9177 - val_loss: 0.3044
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 520s 1s/step - accuracy: 0.9283 - loss: 0.2300 - val_accuracy: 0.9230 - val_loss: 0.2656
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 553s 1s/step - accuracy: 0.9337 - loss: 0.1972 - val_accuracy: 0.9253 - val_loss: 0.2433
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 510s 1s/step - accuracy: 0.9372 - loss: 0.1822 - val_accuracy: 0.9217 - val_loss: 0.2504
Test accuracy: 92.63%


In [5]:
predictions = model.predict(x_test[:10], verbose=0)
for i, prediction in enumerate(predictions):
    actual = CLASS_NAMES[y_test[i]]
    predicted = CLASS_NAMES[np.argmax(prediction)]
    print(f"Image {i + 1}: Actual = {actual}, Predicted = {predicted}")

Image 1: Actual = airplane, Predicted = airplane
Image 2: Actual = automobile, Predicted = automobile
Image 3: Actual = automobile, Predicted = automobile
Image 4: Actual = airplane, Predicted = airplane
Image 5: Actual = truck, Predicted = truck
Image 6: Actual = truck, Predicted = truck
Image 7: Actual = airplane, Predicted = airplane
Image 8: Actual = truck, Predicted = truck
Image 9: Actual = airplane, Predicted = airplane
Image 10: Actual = truck, Predicted = truck
